# Graph Knowledge Network for Agentic Work
## Architecture & Implementation Blueprint

> **A markdown-only design notebook.** This document specifies how to construct a **graph knowledge network** that serves as the substrate for autonomous, self-improving agent work. It is the synthesis of several major projects already in this workspace:
>
> 1. **GraphRAG Survey Synthesis** — three foundational papers (Peng et al. 2024, arXiv:2408.08921; Han et al. 2025, arXiv:2501.00309; Yang et al. 2026, arXiv:2602.05665)
> 2. **20260625 ScientificInfrastructure** — the folder-graph knowledge management system (structurelist.json, input/output.json, subject notebooks, growth procedures)
> 3. **20260706 Obsidian notebook** — Obsidian-style knowledge graphs (nodes, wikilink edges, local traversal, vault export)
> 4. **PDF extraction pipeline** — pypdf text extraction and OpenDataLoader-PDF markdown/JSON extraction of the three surveys

---

# §1 — Motivation: From RAG to GraphRAG to Agent Memory

## 1.1 The Three Limits of Plain RAG

Peng et al. (2024) identify three fundamental limitations of retrieval-augmented generation when confronted with real-world complexity:

| # | Limitation | Consequence |
|---|---|---|
| 1 | **Neglecting relationships** | Retrieves passages independently; misses structured relational knowledge (e.g., citation links *between* papers) that semantic similarity alone cannot represent |
| 2 | **Redundant information** | Concatenated snippets make prompts excessively long → the *lost-in-the-middle* dilemma degrades LLM attention |
| 3 | **Lacking global information** | Retrieves only a subset of documents; fails at Query-Focused Summarization (QFS) and global-context tasks |

## 1.2 What Graphs Give Us

Graphs encode relational knowledge through their intrinsic *nodes-connected-by-edges* nature. Unlike text (1D sequences) or images (2D grids):

| Property | What it enables |
|---|---|
| **Heterogeneous relations** | Entities connected by semantic, temporal, causal, and logical relations across domains |
| **Multi-hop dependencies** | Reasoning chains across relational steps: Disease →[indication]→ Drug ←[target]← Gene |
| **Hierarchical structure** | From fine-grained triples at leaves to global community summaries at the top |
| **Temporal dynamics** | Time-aware edges capture event sequences, state transitions, knowledge evolution |

Graph data also *abstracts and summarizes* text — retrieved subgraphs and communities drastically shorten input length, addressing verbosity and enabling global views (Peng: *G-Retrieval* of nodes/triples/paths/subgraphs; Microsoft GraphRAG: Leiden communities + pre-computed summaries).

## 1.3 Three Structural Differences: RAG vs GraphRAG (Han et al. 2025)

1. **Unified vs. diverse-formatted information** — text/images have transferable units; graphs come in heterogeneous formats (document chunks, KG triples, molecular complexes), so *one-size-fits-all* GraphRAG is impossible.
2. **Independent vs. interdependent information** — RAG chunks are indexed independently; graph nodes are connected, enabling multi-hop traversal and structure-aware organization.
3. **Domain invariance vs. domain-specific information** — relational patterns differ per domain (homophily works for papers, fails for airport hubs); GraphRAG must be designed per domain (the 10-domain taxonomy).

**Implication for us:** our knowledge network is primarily a *document/knowledge graph* over research subjects — we adopt the document-graph and knowledge-graph techniques, and keep the design domain-aware rather than assuming a universal recipe.

## 1.4 Why Agents Need Memory (Yang et al. 2026)

LLM-based agents fail at long-horizon work due to (i) knowledge cutoff, (ii) tool incompetence, (iii) performance saturation — repeated errors without learning. Memory addresses this, enabling **four objectives**:

1. **Personalization** — capture user preferences, histories, task-specific contexts.
2. **Long-term reasoning beyond the context window** — unbounded external storage, continuous learning, post-deployment experience accumulation.
3. **Knowledge accumulation** — facts persist across sessions.
4. **Self-evolution** — iterative reasoning that improves decision strategies over time.

## 1.5 Memory Taxonomy & the Unifying Insight

| Dimension | Types |
|---|---|
| Temporal span | Short-term (volatile, current context) vs. long-term (persistent) |
| Content type | Knowledge memory (static facts) vs. experience memory (dynamic personal log) |
| Structure | Non-structural (linear buffers, vector DBs) vs. structural (graph-based) |

**The unifying perspective:** traditional memory forms are *degenerate cases of graph memory* — a linear buffer is a chain in a graph; a vector store is a fully-connected similarity-weighted graph. A graph therefore provides the *general* framework: it models **how things are connected**, not just what happened.

## 1.6 The Synthesis: One Network, Two Faces

This notebook unifies the two faces of graph knowledge:

- **Face A — External GraphRAG**: the network as a retrievable knowledge base (Peng's G-Indexing → G-Retrieval → G-Generation; Han's five components).
- **Face B — Internal Agent Memory**: the network as the agent's evolving cognitive architecture (Yang's memory life cycle: *extraction → storage → retrieval → evolution*).

The same graph serves both: what the agent retrieves for a task (Face A) is exactly what the agent evolves after the task (Face B). This is the central design decision of this document.

---

# §2 — Design Principles & Requirements

Every architectural decision below is derived from an existing, working artifact in this workspace. The table records the provenance.

| # | Principle | Provenance | Requirement this imposes |
|---|---|---|---|
| P1 | **Node = knowledge unit with rich metadata** | Obsidian notebook (notes = nodes with label, category, description); ScientificInfrastructure (subject folders) | Every node carries a canonical ID, human-readable name, category, description/summary, and content store |
| P2 | **Typed, directed edges** | Obsidian (relationship types: `described_by`, `depends_on`, `example_of`…); ScientificInfrastructure (input/output) | Every edge has a relation type; edges are traversable in both directions |
| P3 | **Bidirectional consistency** | ScientificInfrastructure §4.3a: ∀X,Y: Y ∈ X.output ⇔ X ∈ Y.input | Any edge creation/removal updates both sides atomically; a validator runs after every mutation |
| P4 | **Global registry = backward compatibility** | ScientificInfrastructure `structurelist.json` (folderid ↔ entryname, immutable) | A single global index exists; legacy folders/notebooks keep working unmodified |
| P5 | **Traceability & audit** | §4.4a Version Control Log; §4.1 info.txt; §4.5a reference naming | Every node logs revisions; every change is attributable to an agent run |
| P6 | **Cell-level modularity** | §4.4b (one `##` per cell) | Content artifacts are granular and machine-editable (this notebook itself follows it) |
| P7 | **Context-bounded operation** | Obsidian `traverse_graph(…, depth)`; Han's ToG local reasoning; lost-in-the-middle | Agents never see the whole graph; they see a **depth-3 local graph** |
| P8 | **Hybrid retrieval: vectors + structure** | Peng (indexing strategies); Han (retriever taxonomy) | Encoder layer (embeddings/vector index) + graph traversal layer, orchestrated per query |
| P9 | **Guarded growth** | §4.7c (dedup, ≤5 new subjects/run); §4.5a (≤5 refs/subject, ≤50/run) | All growth operations pass quality gates and respect per-run limits |
| P10 | **Self-evolution** | Yang §VII (consolidation, graph reasoning, reorganization, exploration) | The graph has explicit update operators beyond append: merge, infer, prune, rewire, probe |
| P11 | **Agent-ready interface** | Han (agent-based retrieval); Yang (agent-based memory operators) | The network exposes tool functions the agent can call; the agent is swappable (default: DeepSeek Chat Completions) |
| P12 | **Local-first & open formats** | Obsidian philosophy; ScientificInfrastructure (JSON/txt); workspace scripts (PDF→txt/md) | Plain files, JSON, Markdown; no proprietary store required; everything human-browsable |

## 2.1 Non-Negotiable Requirements (from the user brief)

1. A **global graph** exists for backward compatibility with all existing infrastructure.
2. **Most importantly**: each node builds a **local graph of its depth-3 connected neighborhood**, and agents operate on those local graphs.
3. **Encoder-like capability**: vector-based RAG extraction into nodes (embeddings + vector index).
4. Nodes **operate on local graphs** to obtain further information (traversal + reasoning).
5. The graph must be able to **grow via recursive self-improvement** (informed by ScientificInfrastructure §4.7 and Yang §VII).
6. A **DeepSeek-powered agent** (Chat Completions API) is ready to facilitate network growth.

## 2.2 Design Tensions Resolved

| Tension | Resolution |
|---|---|
| Global completeness vs. context window | Global graph stored; only depth-3 local graphs enter agent context |
| Rich metadata vs. schema rigidity | Core fields fixed (P1–P4); optional extension fields allowed |
| Agent autonomy vs. runaway growth | Autonomy bounded by §4.7c limits + bidirectional consistency + version control |
| Vector similarity vs. relational truth | Both are kept; similarity proposes candidates, graph traversal verifies/explains |
| Memory growth vs. noise | Internal evolution includes pruning, merging, forgetting (Yang §VII-A3) |

---

# §3 — System Architecture Overview

## 3.1 Layered View

```mermaid
graph TD
    subgraph L1["Layer 0 · Global Graph (backward compatible)"]
        G["Global graph G = (V, E)"]
        R["structurelist.json registry"]
        F["Legacy folders / notebooks / Obsidian vault"]
        G --- R
        G --- F
    end

    subgraph L2["Layer 1 · Node + Local Graph"]
        N1["Node u (folder/note/paper)"]
        LG["Local graph: depth-3 ego network of u"]
        N1 --> LG
        G -. materialize .-> LG
    end

    subgraph L3["Layer 2 · Encoder Layer (vector RAG)"]
        ENC["Encoder: chunk → embed → vector index"]
        ENC --> N1
        ENC --> LG
    end

    subgraph L4["Layer 3 · Agent"]
        A["DeepSeek agent (chat completions + tools)"]
        Q["Task / query on node u"]
        Q --> A
        A -->|vector retrieval| ENC
        A -->|graph traversal| LG
        A -->|generate / decide| ANS["Answer · action · artifact"]
    end

    subgraph L5["Layer 4 · Growth & Self-Improvement"]
        GR["Growth ops: create · link · merge · prune · infer · probe"]
        ANS --> GR
        GR -->|update| G
        GR -->|validate| R
        GR -->|refresh| LG
    end
```

## 3.2 The Five Layers

| Layer | Name | Responsibility | Key artifacts |
|---|---|---|---|
| 0 | **Global Graph** | Store the authoritative network; registry; legacy compatibility | `structurelist.json`, folders `1/…N/`, `input.json`/`output.json`, Obsidian vault, NetworkX `DiGraph` |
| 1 | **Node + Local Graph** | Materialize a bounded working context for each task | Depth-3 ego network (induced subgraph), node summaries |
| 2 | **Encoder Layer** | Convert node content to vectors; index; similarity retrieval | Embeddings store, vector index, chunk store |
| 3 | **Agent** | Query processing, retrieval orchestration, reasoning, generation, tool calls | DeepSeek client, tool functions, prompts |
| 4 | **Growth** | Evolve the graph: external expansion + internal self-evolving + exploration | Growth operators, validators, limits, version logs |

## 3.3 Canonical Data Flow

1. **Anchor** — the task targets a node $u$ (resolved by ID, name, or vector query).
2. **Materialize** — extract the **depth-3 local graph** $\mathcal{L}(u)$ = induced subgraph over all nodes within 3 hops of $u$ (§5).
3. **Encode** — embed the query and the node content; retrieve top-$k$ chunks/nodes from the vector index (§6).
4. **Operate** — the agent reasons over $\mathcal{L}(u)$ + retrieved chunks using the retrieval operators and organizer (§7).
5. **Generate** — DeepSeek produces the answer/action with verbalized graph context (§8).
6. **Evolve** — actions that change knowledge (new subjects, links, references, summaries, inferred edges) are applied through growth operators with full validation (§9).
7. **Loop** — the updated graph feeds the next task; the network is the agent's memory (Face B).

## 3.4 Why This Shape Works

- **Backward compatibility** (Layer 0) means we never break the existing ScientificInfrastructure or Obsidian workflows.
- **Bounded context** (Layer 1) keeps prompts small, avoiding lost-in-the-middle and making each agent step cheap and parallelizable.
- **Vectors + structure** (Layer 2 + 1) mirrors the proven HybridRAG / Microsoft GraphRAG pattern: similarity proposes, structure confirms and explains.
- **Agent-mediated growth** (Layers 3 + 4) implements the recursive self-improvement loop that the user demands, with the guardrails from §4.7 to keep growth healthy.

---

# §4 — The Global Graph: Backward-Compatible Backbone

## 4.1 What the Global Graph Is

The global graph $G = (V, E, X)$ is the **authoritative, complete, persistent** network. It is *not* the runtime working set of any agent — it is the store that all local graphs are materialized from, and the target that all growth operations write back to.

It is **backward compatible** because it is represented as — and continuously synchronized with — the existing artifacts in this workspace:

| Existing artifact | Role in global graph |
|---|---|
| `structurelist.json` | The registry: `folderid` ↔ `entryname` (immutable one-to-one mapping) |
| `N/` subject folders (info.txt, subject.ipynb, references/) | Node content stores |
| `N/input.json` | Incoming edges (prerequisites) |
| `N/output.json` | Outgoing edges (downstream subjects) |
| Obsidian vault (`*.md` with `[[wikilinks]]`) | A human-readable projection of the same graph |
| NetworkX `DiGraph` / optional graph DB (Neo4j, NebulaGraph) | The in-memory or server-side working copy |

**Invariant:** the folder files, the Obsidian vault, and the in-memory graph are *always* generated from one source of truth and kept in sync (see §10.4–10.5).

## 4.2 Node Model

A node unifies the ScientificInfrastructure *subject* and the Obsidian *note*:

```json
{
  "node_id": 6,
  "entryname": "低空经济",
  "category": "subject",
  "description": "Low-altitude economy: UAM, UAV logistics, LAWNs, ISAC...",
  "info_file": "6/info.txt",
  "notebook": "6/subject.ipynb",
  "references": ["6/references/arXiv_2504.09153_....pdf"],
  "embedding_ref": "vec://6@latest",
  "created": "2026-06-25",
  "version": "v2.0"
}
```

In the Obsidian projection, the same node is a note whose filename is `entryname` and whose body contains `[[wikilinks]]` for every edge (§7 of the Obsidian notebook).

## 4.3 Edge Model & the Bidirectional Invariant

An edge $u \to v$ (relation $r$) is stored **twice**, as two halves:

- `u/output.json` gains `{"outputid": v, "entryname": name_v}`
- `v/input.json` gains `{"inputeid": u, "entryname": name_u}`

**Bidirectional consistency (ScientificInfrastructure §4.3a):**

$$\forall X, Y:\quad Y \in X.\text{output} \iff X \in Y.\text{input}$$

A validator runs after every mutation; a failed check aborts the operation and rolls back. Example edge set from the existing build:

```mermaid
graph TD
    A["1: Linear Algebra"] --> C["3: Deep Learning Basics"]
    B["2: Calculus"] --> C
    C --> F["6: 低空经济"]
    D["4: Wireless Communications"] --> F
    E["5: UAV and Drone Technology"] --> F
    E --> G["7: 低空经济应急"]
    F --> G
```

## 4.4 Global Operations (the minimal API)

| Operation | Signature | Guardrails |
|---|---|---|
| `register_node` | `(entryname, category, description) → node_id` | Dedup check against registry (§4.7b); `new_id = 1 + max(folderid)` |
| `link` | `(u, v, relation) → edge` | §4.3a bidirectional write; relation in allowed vocabulary |
| `unlink` | `(u, v) → ∅` | §4.3a bidirectional removal |
| `validate` | `() → report` | Registry completeness; bidirectional consistency; connectivity stats |
| `materialize_local` | `(u, depth=3) → local graph` | Bounded expansion (§5) |
| `export` | `(format) → artifact` | Obsidian vault / markdown / graphml / JSON |

## 4.5 Why 
 Matters Here

The ScientificInfrastructure folders, notebooks, and version-control logs are already a working system with real content (7 subjects, ~50 papers, v1.0→v2.0 notebooks). The graph network *adopts* this system as Layer 0 rather than replacing it. Any legacy tool that reads folders/JSON keeps working; the new layers (encoder, local graphs, agent) simply read and write through the same files. This is the cheapest, safest way to stand up a knowledge network on top of existing assets.

---

# §5 — Local Graphs: Depth-3 Neighborhoods (the Operational Core)

> **Most important requirement of the system:** agents never see the global graph. Each node materializes a **local graph** of its nearby **depth-3 connected neighbors**, and all work happens there.

## 5.1 Definition

For a node $u$, the **local graph of depth $k$** is the ego network:

$$\mathcal{L}_k(u) = G\big[\{v \in V : \text{dist}(u, v) \le k\}\big]$$

i.e., the **induced subgraph** over all nodes whose shortest-path distance from $u$ is at most $k$, including all edges among them. The system default is $k = 3$.

This is exactly what the Obsidian notebook's `traverse_graph(G, start, depth)` already does with BFS, and what Han et al.'s ToG does with beam search over relational paths — but here it is *materialized once per task* and becomes the agent's working memory.

## 5.2 Why Depth 3?

1. **Three hops cover the canonical multi-hop reasoning chains** in the GraphRAG literature: MetaQA's 1/2/3-hop benchmarks; the disease→drug→gene example (Han); ToG reasoning paths; RoG planned relation paths. Depth-3 captures essentially all *reasoning-grounded* retrieval while excluding the long-tail noise.
2. **Bounded context**: worst-case size grows like the 3rd power of degree, but real knowledge graphs are sparse. For a node of average degree $d \approx 5$–10, $\mathcal{L}_3(u)$ is typically 100–1000 nodes — already too many for raw text, which is why the encoder layer (§6) and pruning (§5.6) are needed. Depth-4+ explodes prompt length and re-introduces lost-in-the-middle.
3. **Locality of agentic work**: a task on subject $u$ (e.g., 
) is grounded in its prerequisites ($v \leftarrow u$), its applications ($u \to w$), and its cross-links — all within 1–3 hops. Deeper connections are reached *iteratively*: after acting on $\mathcal{L}_3(u)$, the agent can re-anchor on a neighbor and materialize *its* local graph (multi-round retrieval, Yang §VI).

## 5.3 Extraction Algorithm

```text
Input:  G (global graph), u (anchor node), k = 3
Output: L = (V_L, E_L)  # induced subgraph

1. visited = {u}; frontier = {u}
2. for depth in 1..k:
3.     next = ∅
4.     for node in frontier:
5.         for (nbr, rel) in out_edges(node) ∪ in_edges(node):   # both directions
6.             if nbr not in visited:
7.                 visited.add(nbr); next.add(nbr)
8.     frontier = next
9. V_L = visited
10. E_L = {(a,b,r) in E : a,b in V_L}      # keep ALL internal edges, incl. shortcuts
11. annotate each node with: entryname, category, summary, embedding_ref
12. annotate each edge with: relation, weight, direction
13. return L
```

Notes: expansion follows **both** incoming and outgoing edges (undirected hops) because a prerequisite (`input`) is as relevant as a downstream (`output`); the induced subgraph keeps shortcut edges so the agent sees the full relational structure, not just BFS trees.

## 5.4 What the Local Graph Carries

Each local graph ships with **summarized payloads** so the agent gets structure + content without dumping full documents:

| Element | Payload |
|---|---|
| Node | `node_id`, `entryname`, `category`, 1–2 sentence summary, `embedding_ref` (id, not vector) |
| Edge | `(source, target, relation, weight)` |
| Attachments | pointers to `references/` PDFs, notebook sections, Obsidian notes |
| Graph stats | `|V_L|, |E_L|, density, diameter`, community ids (optional Leiden) |

## 5.5 Granularity: Nodes → Triples → Paths → Subgraphs

Peng et al. define the retrieval granularity spectrum; the local graph supports all of them:

| Granularity | How the agent uses $\mathcal{L}_3(u)$ |
|---|---|
| **Node** | Top-$k$ relevant nodes by vector similarity or PageRank within $\mathcal{L}_3(u)$ |
| **Triple** | Verbalized `(head, relation, tail)` rows for compact context |
| **Path** | Shortest / highest-weight paths between query entities (multi-hop reasoning) |
| **Subgraph** | The whole $\mathcal{L}_3(u)$, or a community/PCST-pruned sub-subgraph for QFS |
| **Hybrid** | LLM agent decides granularity per query (Han: agent-based retrieval) |

## 5.6 Local-Graph Optimization (Pruning Before the Prompt)

To keep context small and signal high (Organizer: *graph pruning*, Peng/Han):

1. **PageRank / significance pruning** — drop nodes with score below threshold (also the forgetting operator, §9.2).
2. **Query-relevance pruning** — drop nodes whose embedding similarity to the query is low (QA-GNN style).
3. **Prize-Collecting Steiner Tree (PCST)** — extract the cheapest connected subgraph spanning query entities (G-Retriever).
4. **Community subgraph** — if $\mathcal{L}_3(u)$ is large, run Leiden and include only the communities touching the anchor + query entities (Microsoft GraphRAG).
5. **Path-focused extraction** — for multi-hop queries, keep only the top-$k$ shortest paths and their neighborhood.

## 5.7 Local Graphs as the Agent's Working Memory

In Yang et al.'s terms, $\mathcal{L}_3(u)$ is the **working memory** (active subgraph of currently attended nodes) materialized from **long-term semantic/experience memory** (the global graph). After each task, the working memory dissolves; the *changes* (new facts, lessons, links) are consolidated back into the global graph by the growth layer (§9). This mirrors human cognition: narrow attention during work, consolidation after.

PDF ──► extract_pdfs.py (pypdf) ──► extracted/*.txt          (page-marked plain text)

---

# §7 — The Agent Operating on Local Graphs

> **Requirement:** *
*

Once a local graph $\mathcal{L}_3(u)$ is materialized and encoded, the agent runs a GraphRAG pipeline **restricted to that local graph**. This section specifies the pipeline, the retrieval operators, and the loop.

## 7.1 The Five-Component Pipeline on a Local Graph (Han et al.)

$$A = \Omega_{Gen}\big(\hat{q},\; \Omega_{Org}(\hat{q},\; \Omega_{Ret}(\hat{q},\; \mathcal{L}_3(u)))\big)$$

| Component | Local-graph adaptation |
|---|---|
| **Query Processor** $\Omega_{Proc}$ | NER over the query → map to node IDs in $\mathcal{L}_3(u)$; relation matching to edge types; optional decomposition into sub-queries; query expansion via 1-hop neighbors of mentioned entities |
| **Retriever** $\Omega_{Ret}$ | Hybrid: vector top-$k$ (§6.6) + graph traversal (BFS/DFS, $k$-hop, shortest paths, PCST) — all inside $\mathcal{L}_3(u)$ |
| **Organizer** $\Omega_{Org}$ | Prune (relevance/PageRank), rerank (cross-encoder or hybrid score), augment (add query node), verbalize (triples → text) |
| **Generator** $\Omega_{Gen}$ | DeepSeek Chat Completions with verbalized context (§8) |
| **Data Source** $\mathcal{G}$ | $\mathcal{L}_3(u)$ (bounded), plus node content pulled via encoder layer |

## 7.2 The Six Base Retrieval Operators (Yang et al.)

| Operator | Mechanism inside $\mathcal{L}_3(u)$ | When to prefer |
|---|---|---|
| **Similarity-based** | Embed query → top-$k$ node/chunk vectors | Fuzzy, open-ended questions |
| **Rule-based** | Deterministic filters: `category == 'application'`, references of type arXiv, section headings | Precise, verifiable extraction |
| **Temporal-based** | Rank edges/nodes by recency (created, updated, paper date) | 
, survey updates |
| **Graph-based** | Intra-layer neighborhood expansion; cross-layer traversal via abstraction edges | Multi-hop 
 |
| **RL-based** | (Optional) learned policy for retrieval budget — deferred; heuristic first | Large-scale production |
| **Agent-based** | LLM decides granularity, next hop, stop condition (ToG-style beam reasoning) | Complex, open-ended research tasks |

## 7.3 Three Enhancement Strategies

1. **Multi-round retrieval** — act on $\mathcal{L}_3(u)$, then re-anchor on the most promising neighbor $v$ and materialize $\mathcal{L}_3(v)$; iterate until evidence is consistent or $\le R$ rounds (default 3). This is how depth-3 locality composes into arbitrarily deep reasoning.
2. **Post-retrieval (generate-then-retrieve)** — first have the LLM produce an intermediate representation (topic descriptor, hypothesized entities/relations), then retrieve against *that* — robust to query phrasing (SimGRAG-style).
3. **Hybrid-source retrieval** — coordinate internal graph memory with external sources (arXiv search API, PDF downloads, the web); conflict resolution rules: external evidence overrides stale internal unless newer internal wins (temporal rule).

## 7.4 The Organizer: From Local Graph to Prompt

1. **Prune** — drop low-relevance, low-PageRank nodes (§5.6).
2. **Rerank** — hybrid score (§6.6) or cross-encoder.
3. **Verbalize** — choose a graph-to-text format (§7.5).
4. **Budget** — token budget for context (e.g., 8k tokens context out of a 64k window), enforced by truncation order: anchor node → directly connected triples → paths → summaries → raw chunks.

## 7.5 Graph-to-Text Formats (choose per query type)

| Format | Example | Best for |
|---|---|---|
| Triple rows | `(低空经济, has_topic, LAWN)` | Fact-checking, entity-centric QA |
| Edge table / adjacency | per-node neighbor list | Exhaustive structural questions |
| Path strings | `UAV → LAWN → ISAC → Emergency Comms` | Multi-hop reasoning |
| Community summary | Leiden cluster → LLM summary | Global / summarization questions |
| Node-sequence | BFS/DFS order | Compact context |

## 7.6 The Node-Operation Loop (pseudocode)

```text
def operate_on_node(u, task):
    L  = materialize_local(u, depth=3)          # §5
    ctx = encode_and_retrieve(L, task)          # §6: vector top-k + structure
    plan = agent_reason(L, ctx, task)           # query processing + operator choice
    for step in plan:
        sub = execute_step(L, ctx, step)        # traversal / retrieval / tool call
        ctx = ctx.merge(verbalize(sub))         # organize + verbalize
        if stopping_criterion(L, sub): break    # adaptive stop (ToG-style)
    answer = deepseek_generate(ctx, task)       # §8
    record_evidence(answer, ctx)                # provenance: which nodes/edges were used
    return answer
```

**Provenance is mandatory:** every answer records which nodes/edges of the local graph grounded it. This becomes the *experience memory* that feeds self-improvement (§9.3) and enables verifiable generation (§12).

---

# §8 — DeepSeek-Powered Agent: Chat Completions API

> **Requirement:** *
*

## 8.1 API Setup (OpenAI-Compatible)

DeepSeek exposes an OpenAI-compatible Chat Completions endpoint, so the standard `openai` SDK works with a custom `base_url`:

```python
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],   # never hard-code secrets
    base_url="https://api.deepseek.com",      # OpenAI-compatible endpoint
)

MODEL = "deepseek-chat"       # general agentic work (DeepSeek-V3 class)
MODEL_REASONER = "deepseek-reasoner"  # hard reasoning / planning tasks

def chat(messages, tools=None, temperature=0.3):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools or [],          # function calling (OpenAI schema)
        tool_choice="auto",
        temperature=temperature,
    )
    return resp.choices[0].message
```

Key parameters to tune per task: `temperature` (0.2–0.3 for graph/JSON operations, 0.7+ for creative synthesis), `max_tokens` (bounded by the organizer budget, §7.4), `tools` (function-calling schema, §8.2), and `stream` for long generations.

## 8.2 The Tool Interface: the Agent's Hands on the Network

Each tool is a callable backed by a global-graph or local-graph operation; the LLM selects them via function calling:

| Tool | Backed by | Purpose (growth-oriented) |
|---|---|---|
| `get_local_graph(node_id, depth=3)` | §5 | Materialize working context |
| `search_nodes(query, k)` | §6 | Vector RAG extraction into nodes |
| `read_node(node_id, part)` | §4 | info.txt / notebook section / references list |
| `register_node(entryname, category, description)` | §4.4 | Create a new subject/note (with dedup + registry check) |
| `link(u, v, relation)` | §4.3 | Add edge with bidirectional consistency |
| `unlink(u, v)` | §4.3 | Remove edge bidirectionally |
| `summarize_node(u)` / `summarize_community(L)` | §7.5 | Generate/refresh summaries (consolidation) |
| `infer_edges(u)` | §9.2 | Latent link prediction in local graph |
| `add_reference(u, arxiv_id_or_url)` | §4.5a | Acquire literature (limits enforced) |
| `update_notebook(u, section_md)` | §4.4 | Machine-edit subject.ipynb sections |
| `probe_gap(u)` | §9.3 | Active inquiry: report missing nodes/ambiguous edges |
| `validate()` | §4.4 | Run consistency + health checks before finishing a run |

## 8.3 The Agent Loop

```text
while task pending:
    1. PERCEIVE  — materialize L = local_graph(u, 3); pull encoded ctx   (§5, §6)
    2. REASON    — chat(L, ctx, task, tools) → next tool call or answer   (§7)
    3. ACT       — execute tool call; apply growth op with guardrails     (§4, §9)
    4. OBSERVE   — refresh local graph; validator runs; collect result
    5. LEARN     — write experience record: (task, evidence nodes, outcome, lesson)
                  → new nodes/edges or weights (episodic memory, §9.3)
```

## 8.4 Prompt Templates (core four)

**A. Node-description prompt** (fills `info.txt`):

```text
You are maintaining a research knowledge graph. Write a 3–5 sentence
description of the node "{entryname}" based on: {local_graph_summary}
{evidence}. Include (a) what it is, (b) key subtopics, (c) its role
between its inputs {inputs} and outputs {outputs}. Output plain text only.
```

**B. Link-proposal prompt** (creates edges with a relation from the vocabulary):

```text
Given the local graph and the new content {content}, propose up to 5
new edges (source, target, relation) using only these relation types:
{vocab}. For each, give a one-sentence justification. Skip duplicates.
Return JSON: [{"source":..., "target":..., "relation":..., "why":...}]
```

**C. Growth-decision prompt** (agent-initiated subject creation, §4.7b):

```text
The topic {topic} appeared while working on node {u}. Decide:
1) DEDUP: is it semantically covered by an existing node? List closest
   candidates and similarity scores.
2) If new: propose entryname, category, description, and which existing
   nodes should be its inputs.
Return JSON with decision + rationale. Respect the ≤5 new subjects/run cap.
```

**D. Consolidation prompt** (internal self-evolving, §9.2):

```text
Consolidate the episodes {episodes} into generalized knowledge for node
{u}. 1) Merge similar observations into a canonical fact (with confidence).
2) Detect contradictions; resolve by recency + confidence. 3) Propose
inferred edges. Return JSON: {merge:[], conflicts:[], new_edges:[]}.
```

## 8.5 Guardrails for the Agent

1. **Limits**: ≤5 new subjects/run (§4.7c); ≤5 references per subject per run, ≤50 total (§4.5a); ≤5 proposed edges per prompt (template B).
2. **Consistency**: every `link`/`unlink` runs the §4.3a validator; failures abort.
3. **Dedup**: every `register_node` consults the registry + embeddings.
4. **Audit**: every mutation appends to the affected node's Version Control Log (§4.4a) and to the run log.
5. **Secrets**: API key from environment variable only; never in notebooks or JSON.

---

# §9 — Recursive Self-Improvement & Graph Growth

> **Requirement:** *
20260625
*

Growth = **external expansion** (new nodes/edges from the world) + **internal self-evolving** (consolidation, reasoning, reorganization) + **external self-exploration** (feedback-driven adaptation, active inquiry). The loop composes them into recursive self-improvement.

## 9.1 External Expansion (ScientificInfrastructure §4.7)

### 9.1a Agent-Initiated New Subject Creation (§4.7b)

| Step | Action | Guardrail |
|---|---|---|
| 1 | Dedup check: search `structurelist.json` + embeddings for semantic matches | If close match → link, don't create |
| 2 | `new_id = 1 + max(folderid)` | Unique |
| 3 | Register in `structurelist.json` | Registry first, before any references (§4.7c) |
| 4 | Create full folder: `info.txt`, `input.json`, `output.json`, `subject.ipynb`, `references/` | §4.1–§4.6; VCL seeded (§4.4a) |
| 5 | `input.json` lists motivating subjects | §4.3a: mirror into their `output.json` |
| 6 | Acquire ≤5 references (arXiv naming convention) | §4.5a |
| 7 | Encoder: chunk + embed new content; index | §6 |

### 9.1b Splitting an Overgrown Subject (§4.7a)

**Trigger:** >~500 references in one folder. **Procedure:** choose the knowledge slice → new node with next id → register → build folder → move references + notebook sections → edges (parent-child) → bidirectional consistency → VCL in both notebooks. Splitting keeps local graphs shallow and retrieval fast.

### 9.1c Limits (§4.7c, §4.5a)

| Constraint | Limit |
|---|---|
| New subjects per run | ≤ 5 |
| References per subject per run | ≤ 5 |
| Total references per run | ≤ 50 |
| Duplicates | none (semantic dedup) |
| Edge consistency | ∀X,Y: Y ∈ X.output ⇔ X ∈ Y.input |

## 9.2 Internal Self-Evolving (Yang §VII-A)

Closed-loop refinement without external input — the graph improves itself:

| Mechanism | Operator | Example |
|---|---|---|
| **Memory consolidation** | Generalization via graph merging | Merge similar episode subgraphs into a canonical skill/fact node (Mem0-style semantic gating: only merge if information gain passes threshold) |
| **Inference & conflict resolution** | Consistency repair | A ⇒ B and C ⇒ ¬B detected → resolve by confidence + recency; update structure |
| **Graph reasoning** | Latent link prediction | From (A →cause B) and (B →cause C) infer (A →leads_to C); fill sparse areas of the graph |
| **Inductive enrichment** | Path-based attribute derivation | Traverse local graph to derive new attributes/high-order relations |
| **Graph reorganization** | Significance-based pruning | PageRank/decay-based forgetting of rarely-accessed nodes (keeps depth-3 local graphs clean) |
| **Topology optimization** | Edge rewiring/shortcuts | Increase weights or add shortcuts between frequently co-retrieved concepts → faster future retrieval |

## 9.3 External Self-Exploration (Yang §VII-B)

Grounding and expansion through interaction with the world:

| Paradigm | Mechanism | Tool from §8.2 |
|---|---|---|
| **Feedback-driven adaptation (reactive)** | Successful trajectories crystallize into skills; failures → comparative analysis → 
 edges; memory-management policy itself learnable (RL over add/delete actions, Memory-R1-style) | `update_notebook`, `infer_edges` |
| **Active inquiry (proactive)** | Detect uncertainty/missing nodes → generate queries to fill gaps (ProMem-style); autonomously generate sub-tasks that traverse unexplored states (AgentEvolver-style); optional swarm of parallel sub-agents ingesting diverse perspectives (KIMI-K2.5-style) | `probe_gap`, `register_node`, `add_reference` |

**Graph-specific exploration suggestion (from Yang):** prioritize sparsely-connected clusters and bridge disconnected subgraphs — this is exactly what `probe_gap` should target, guided by connectivity stats of the global graph.

## 9.4 The Recursive Growth Loop

```mermaid
graph TD
    S["Task on node u"] --> M["Materialize local graph depth-3"]
    M --> O["Agent operates: retrieve + reason + generate"]
    O --> E["Extract evidence & lessons"]
    E --> C["Consolidate: merge / resolve conflicts / infer edges"]
    C --> X["Explore: probe gaps · acquire refs · create/link nodes"]
    X --> V["Validate: consistency · dedup · limits · VCL"]
    V -->|ok| M
    V -->|fail| C
    V -. re-anchor on neighbor v .-> S
```

**Why this is *recursive* self-improvement:** the graph is both the input and the output of every loop iteration. Iteration $n$ produces better context (denser, cleaner, better-summarized graph) for iteration $n+1$; and the agent's accumulated experience memory makes its decisions (which gaps to probe, which merges to make) better over time. The ScientificInfrastructure already demonstrates the two outer mechanisms (external expansion §4.7, versioned notebook growth §14); Yang et al. supply the inner mechanisms (consolidation/reasoning/reorganization) that turn a growing archive into a *learning* system.

## 9.5 Health Metrics & Termination Conditions

| Signal | Metric | Action |
|---|---|---|
| Graph health | density, avg path length, #components, dedup-hit rate | Rewire (add shortcuts) if avg path length grows; bridge components if >1 |
| Retrieval drift | answer coverage vs subgraph size ratio declining | Prune + consolidate (forgetting) |
| Diminishing returns | new-subject dedup-hit rate → high | Stop creating; focus on consolidation |
| Conflict density | unresolved contradictions | Priority consolidation pass |

A run terminates when: all tasks done, all validators pass, limits respected, and at least one consolidation or exploration op has improved the graph (recorded in VCL).

| Plain-text extraction | `extract_pdfs.py` → `extracted/*_pypdf.txt` | Page-marked text for chunking |

---

# §11 — Schema Specifications

All schemas are JSON-compatible and plain-file-friendly (P12).

## 11.1 Global Registry (`structurelist.json`) — unchanged for backward compatibility

```json
[
  { "folderid": 1, "entryname": "Linear Algebra" },
  { "folderid": 6, "entryname": "低空经济" },
  { "folderid": 7, "entryname": "低空经济应急" }
]
```

## 11.2 Edge Files (`input.json` / `output.json`) — unchanged

```json
// 6/output.json
[ { "outputid": 7, "entryname": "低空经济应急" } ]

// 7/input.json
[ { "inputeid": 6, "entryname": "低空经济" } ]
```

## 11.3 Node Record (network layer; superset of the folder)

```json
{
  "node_id": 7,
  "entryname": "低空经济应急",
  "category": "subject",
  "summary": "Emergency applications of LAE: UAV-SAR, medical delivery, ISAC...",
  "content": { "info": "7/info.txt", "notebook": "7/subject.ipynb", "chunks": "vec://7/chunks" },
  "embedding": { "ref": "vec://7@v2.0", "model": "bge-m3", "dim": 1024 },
  "stats": { "in_degree": 3, "out_degree": 1, "pagerank": 0.042 },
  "timestamps": { "created": "2026-06-25", "updated": "2026-06-25" },
  "version": "v2.0"
}
```

## 11.4 Edge Record

```json
{
  "source": 6,
  "target": 7,
  "relation": "feeds_into",
  "weight": 1.0,
  "evidence": ["vec://6/chunk/42", "6/references/arXiv_2509.11607.pdf"],
  "temporal": { "valid_from": "2026-06-25", "superseded": false },
  "agent_run": "run-2026-08-02-001"
}
```

## 11.5 Local Graph Serialization (what the agent actually consumes)

```json
{
  "anchor": 7, "depth": 3,
  "nodes": [
    { "id": 7, "name": "低空经济应急", "category": "subject", "summary": "..." },
    { "id": 6, "name": "低空经济", "category": "subject", "summary": "..." }
  ],
  "edges": [ { "source": 6, "target": 7, "relation": "feeds_into" } ],
  "paths": [ [7, 6, 5] ],
  "stats": { "n": 4, "m": 3, "density": 0.5, "diameter": 2 },
  "summaries": { "communities": [ { "id": 0, "text": "..." } ] }
}
```

## 11.6 Vector Index Record (per node, chunk-level)

```json
{
  "chunk_id": "vec://7/c023",
  "node_id": 7,
  "section": "## 3. 应急通信网络架构",
  "text": "...",
  "embedding": [0.012, -0.034, "..."],
  "source_ref": { "file": "odl_output/...json", "page": 9, "bbox": [11.2, 44.0, 520.0, 88.5] },
  "ts": "2026-08-02"
}
```

## 11.7 Run / Audit Record (VCL-compatible)

```json
{
  "run_id": "run-2026-08-02-001",
  "agent": "deepseek-chat",
  "actions": [ { "op": "link", "args": { "source": 6, "target": 9 }, "status": "ok" } ],
  "counts": { "new_subjects": 1, "new_edges": 3, "new_refs": 4, "merged": 1, "pruned": 2 },
  "limits": { "subjects_max": 5, "refs_per_subject_max": 5, "refs_total_max": 50 },
  "validator": "PASS"
}
```

## 11.8 Relation Vocabulary (initial; extensible, domain-aware per Han §10-domains)

`feeds_into` · `depends_on` · `example_of` · `generalizes` · `described_by` · `formulated_by` · `applied_in` · `related_to` · `contains` · `used_in` · `extends` · `splits_from` · `supersedes` · `cites` · `evidence_for` · `lesson_from` (experience memory)

| P3 | **GraphRAG engine** | Three surveys + synthesis notebook; extraction pipeline (pypdf, OpenDataLoader-PDF) | Layer 1–3: local-graph materialization, encoder layer, retrieval/organizer/generator pipeline |

---

# §13 — How This Composes the Bigger Projects

The user brief says this *composes a major part of several big projects*. Here is the explicit decomposition and how the network ties them together.

## 13.1 Project Map

| # | Project | Existing assets | Role in the network |
|---|---|---|---|
| P1 | **Scientific Infrastructure** | `structurelist.json`, folders 1–7, input/output.json, subject.ipynb, README spec §4.x | Layer 0 (global graph): the folder-graph *is* the backbone; growth procedures §4.7 are the growth operators; VCL/limits are the governance |
| P2 | **Obsidian knowledge graph** (2D CFT demo) | Node/category/edge model, traverse_graph BFS, wikilink export | The conceptual pattern for nodes/edges and local traversal; the vault export/import gives humans a graph UI |
| P3 | **GraphRAG engine** | Three surveys + synthesis notebook; extraction pipeline (pypdf, OpenDataLoader-PDF) | Layer 1–3: local-graph materialization, encoder layer, retrieval/organizer/generator pipeline |
| P4 | **Self-evolving agent memory** | Yang et al. taxonomy; DeepSeek API | Layer 3–4: the agent + consolidation/reasoning/reorganization/exploration operators |

## 13.2 Integration Matrix (component → projects served)

| Component | P1 | P2 | P3 | P4 |
|---|---|---|---|---|
| Global graph / registry | ● core | ● shared | ● source | ● memory store |
| Local graph depth-3 | ○ optional (navigation aid) | ● core (traverse_graph) | ● core (G-Retrieval) | ● core (working memory) |
| Encoder layer / vector RAG | ○ (index references) | ○ (next-step in obsidian nb) | ● core (indexing) | ● core (memory retrieval) |
| DeepSeek agent + tools | ● (agent builds folders) | ○ | ● (generation) | ● (self-evolution) |
| Growth & self-improvement | ● core (§4.7) | ○ | ○ | ● core (Yang §VII) |

(● = primary, ○ = secondary)

## 13.3 What Each Project Gains

- **P1 gains** a retrieval and reasoning engine over its folders — its static graph becomes queryable and agent-maintainable.
- **P2 gains** a real RAG backend and an agent that grows the 2D CFT vault automatically.
- **P3 gains** a concrete implementation target: the survey *is* the design doc for the engine, and the extractions *are* the seed corpus.
- **P4 gains** a concrete storage substrate and governance (limits, VCL, dedup) that Yang et al.'s memory life cycle needs in practice.

## 13.4 Phase Roadmap

| Phase | Deliverable | Depends on |
|---|---|---|
| **M0 · Adopt** | Sync existing folders/vault into a NetworkX global graph; validator (§4.3a) passes | P1, P2 |
| **M1 · Encode** | Chunk + embed the three surveys into nodes; vector index; `search_nodes` works | P3 pipeline |
| **M2 · Localize** | `materialize_local(u, 3)` + hybrid retrieval (§6.6); MetaQA 3-hop smoke test | M1 |
| **M3 · Agentify** | DeepSeek tool loop (§8); answer + provenance on local graphs | M2, API key |
| **M4 · Evolve** | Growth operators + consolidation + probe_gap; recursive loop (§9.4); VCL everywhere | M3 |
| **M5 · Harden** | Quality gates (§12), Obsidian two-way sync, scaling (FAISS/HNSW, graph DB option) | M4 |

Each phase keeps backward compatibility: M0–M5 never alter the meaning of `structurelist.json` or folder layout.

---

# §14 — Version Control Log

Per ScientificInfrastructure §4.4a, every knowledge notebook records its revision history. Each entry: date, version, author/agent, summary.

### v1.0 — 2026-08-02 (Initial build, Copilot agent)

- Created the markdown-only design notebook *Graph Knowledge Network for Agentic Work*.
- Synthesized three GraphRAG/agent-memory surveys (Peng 2408.08921, Han 2501.00309, Yang 2602.05665) from the workspace extractions (`extracted/`, `odl_output/`) and the 20260731 synthesis notebook.
- Incorporated 20260625 ScientificInfrastructure conventions: §4.3a bidirectional consistency, §4.4a VCL, §4.4b cell modularity, §4.5a reference limits, §4.7 expansion procedures.
- Incorporated the Obsidian notebook's node/edge model, depth-bounded `traverse_graph`, and wikilink vault export.
- Specified the five-layer architecture: global graph (backward compatible) → depth-3 local graphs → encoder/vector-RAG layer → DeepSeek agent → recursive growth loop.
- Specified the DeepSeek Chat Completions integration (OpenAI-compatible), tool interface, prompt templates, and guardrails.
- Specified growth/self-improvement: external expansion (§4.7), internal self-evolving (consolidation, graph reasoning, reorganization), external self-exploration (feedback adaptation, active inquiry).
- Added schemas (§11), evaluation gates (§12), project-composition matrix and roadmap (§13).

---

*Notebook compiled 2026-08-02. Markdown-only by design (§4.4b). Follows the backward-compatible conventions of the ScientificInfrastructure and the Obsidian knowledge-graph philosophy.*